## OMOP Data Quality Dashboard

### [Startup script](https://ohdsi.github.io/DataQualityDashboard/articles/DataQualityDashboard.html)

### TODO:
 - Fill in connection and schema details
 - Figure out how to view results (see bottom of startup script page)

In [0]:
# Install the missing libtirpc development library (if not already done)
system("apt-get install -y libtirpc-dev")

# Set Java environment variables
Sys.setenv(JAVA_HOME = "/usr/lib/jvm/java-8-openjdk-amd64")

# Reconfigure R's Java settings
system("R CMD javareconf")

# Install rJava
install.packages("rJava")

# Install DatabaseConnector
install.packages("DatabaseConnector")

# Load the libraries
library(rJava)
library(DatabaseConnector)

# Test that it works
packageVersion("DatabaseConnector")


# Install DataQualityDashboard from GitHub
install.packages("remotes")
remotes::install_github("OHDSI/DataQualityDashboard")

# Load it
library(DataQualityDashboard)

In [0]:
# Try using the direct Spark Thrift server connection
connectionDetails <- DatabaseConnector::createConnectionDetails(
  dbms = "spark",
  connectionString = "jdbc:hive2://localhost:10000/default",
  pathToDriver = "/tmp/jdbc_drivers"
)

In [0]:
# Install and use sparklyr (which comes pre-installed in Databricks)
library(sparklyr)

# Connect to the local Spark session
sc <- spark_connect(method = "databricks")

# You can now query your tables directly
ehr_union_tables <- dbListTables(sc, schema_name = "ehr_union")
print(ehr_union_tables)

# Test a simple query
sample_data <- dbGetQuery(sc, "SELECT * FROM ehr_union.person LIMIT 5")
print(sample_data)

In [0]:
# Try connecting to SQL Warehouse instead of local Thrift server
connectionDetails <- DatabaseConnector::createConnectionDetails(
  dbms = "spark",
  connectionString = "jdbc:databricks://localhost:443/default;transportMode=http;ssl=1;AuthMech=0;httpPath=/sql/1.0/warehouses/",
  pathToDriver = "/tmp/jdbc_drivers"
)

In [0]:
# Set sqlOnly to TRUE to generate SQL without connecting
sqlOnly <- TRUE  # This bypasses the connection issues

# Create a minimal connectionDetails just for SQL generation
connectionDetails <- DatabaseConnector::createConnectionDetails(
  dbms = "spark",
  pathToDriver = "/"  # Minimal path since we're not actually connecting
)

# Run DQD in SQL-only mode
DataQualityDashboard::executeDqChecks(
  connectionDetails = connectionDetails, 
  cdmDatabaseSchema = cdmDatabaseSchema, 
  resultsDatabaseSchema = resultsDatabaseSchema,
  cdmSourceName = cdmSourceName, 
  cdmVersion = cdmVersion,
  numThreads = numThreads,
  sqlOnly = sqlOnly,  # This is key - generates SQL without connecting
  outputFolder = outputFolder,
  outputFile = outputFile,
  verboseMode = verboseMode,
  checkLevels = checkLevels,
  checkSeverity = checkSeverity,
  tablesToExclude = tablesToExclude,
  checkNames = checkNames
)

In [0]:

# fill out the connection details -----------------------------------------------------------------------
# Spark / Databricks:
# Currently both JDBC and ODBC connections are supported for Spark. Set the connectionString argument to use JDBC, otherwise ODBC is used:
# connectionString. The JDBC connection string (e.g. something like 'jdbc:databricks://my-org.cloud.databricks.com:443/default;transportMode=http;ssl=1;AuthMech=3;httpPath=/sql/1.0/warehouses/abcde12345;').
# user. The user name used to access the server. This can be set to 'token' when using a personal token (recommended).
# password. The password for that user. This should be your personal token when using a personal token (recommended).
# server. The host name of the server (when using ODBC), e.g. 'my-org.cloud.databricks.com')
# port. Specifies the port on the server (when using ODBC)
# extraSettings. Additional settings for the ODBC connection, for example extraSettings = list(HTTPPath = "/sql/1.0/warehouses/abcde12345", SSL = 1, ThriftTransport = 2, AuthMech = 3)


# connectionDetails <- DatabaseConnector::createConnectionDetails(
# dbms = "databricks", 
# user = "",
# password = "",
# server = "",
# port = "", 
# extraSettings = "",
# pathToDriver = ""
# )

cdmDatabaseSchema <- "ehr_union" # the fully qualified database schema name of the CDM
resultsDatabaseSchema <- "omop_dqd" # the fully qualified database schema name of the results schema (that you can write to)
cdmSourceName <- "EHR Union" # a human readable name for your CDM source
cdmVersion <- "5.4" # the CDM version you are targetting. Currently supports 5.2, 5.3, and 5.4

# determine how many threads (concurrent SQL sessions) to use ----------------------------------------
numThreads <- 1 # on Redshift, 3 seems to work well

# specify if you want to execute the queries or inspect them ------------------------------------------
sqlOnly <- FALSE # set to TRUE if you just want to get the SQL scripts and not actually run the queries
sqlOnlyIncrementalInsert <- FALSE # set to TRUE if you want the generated SQL queries to calculate DQD results and insert them into a database table (@resultsDatabaseSchema.@writeTableName)
sqlOnlyUnionCount <- 1  # in sqlOnlyIncrementalInsert mode, the number of check sqls to union in a single query; higher numbers can improve performance in some DBMS (e.g. a value of 25 may be 25x faster)

# NOTES specific to sqlOnly <- TRUE option ------------------------------------------------------------
# 1. You do not need a live database connection.  Instead, connectionDetails only needs these parameters:
#      connectionDetails <- DatabaseConnector::createConnectionDetails(
#        dbms = "", # specify your dbms
#        pathToDriver = "/"
#      )
# 2. Since these are fully functional queries, this can help with debugging.
# 3. In the results output by the sqlOnlyIncrementalInsert queries, placeholders are populated for execution_time, query_text, and warnings/errors; and the NOT_APPLICABLE rules are not applied.
# 4. In order to use the generated SQL to insert metadata and check results into output table, you must set sqlOnlyIncrementalInsert = TRUE.  Otherwise sqlOnly is backwards compatable with <= v2.2.0, generating queries which run the checks but don't store the results.


# where should the results and logs go? ----------------------------------------------------------------
outputFolder <- "output"
outputFile <- "results.json"


# logging type -------------------------------------------------------------------------------------
verboseMode <- TRUE # set to FALSE if you don't want the logs to be printed to the console

# write results to table? ------------------------------------------------------------------------------
writeToTable <- TRUE # set to FALSE if you want to skip writing to a SQL table in the results schema

# specify the name of the results table (used when writeToTable = TRUE and when sqlOnlyIncrementalInsert = TRUE)
writeTableName <- "dqdashboard_results"

# write results to a csv file? -----------------------------------------------------------------------
writeToCsv <- FALSE # set to FALSE if you want to skip writing to csv file
csvFile <- "" # only needed if writeToCsv is set to TRUE

# if writing to table and using Redshift, bulk loading can be initialized -------------------------------

# Sys.setenv("AWS_ACCESS_KEY_ID" = "",
#            "AWS_SECRET_ACCESS_KEY" = "",
#            "AWS_DEFAULT_REGION" = "",
#            "AWS_BUCKET_NAME" = "",
#            "AWS_OBJECT_KEY" = "",
#            "AWS_SSE_TYPE" = "AES256",
#            "USE_MPP_BULK_LOAD" = TRUE)

# which DQ check levels to run -------------------------------------------------------------------
checkLevels <- c("TABLE", "FIELD", "CONCEPT")

# which DQ checks to run? ------------------------------------
checkNames <- c() # Names can be found in inst/csv/OMOP_CDM_v5.3_Check_Descriptions.csv

# which DQ severity levels to run? ----------------------------
checkSeverity <- c("fatal", "convention", "characterization") 

# want to EXCLUDE a pre-specified list of checks? run the following code:
#
# checksToExclude <- c() # Names of check types to exclude from your DQD run
# allChecks <- DataQualityDashboard::listDqChecks()
# checkNames <- allChecks$checkDescriptions %>%
#   subset(!(checkName %in% checksToExclude)) %>%
#   select(checkName)

# which CDM tables to exclude? ------------------------------------
tablesToExclude <- c("CONCEPT", "VOCABULARY", "CONCEPT_ANCESTOR", "CONCEPT_RELATIONSHIP", "CONCEPT_CLASS", "CONCEPT_SYNONYM", "RELATIONSHIP", "DOMAIN") # list of CDM table names to skip evaluating checks against; by default DQD excludes the vocab tables

# run the job --------------------------------------------------------------------------------------
DataQualityDashboard::executeDqChecks(connectionDetails = connectionDetails, 
                            cdmDatabaseSchema = cdmDatabaseSchema, 
                            resultsDatabaseSchema = resultsDatabaseSchema,
                            cdmSourceName = cdmSourceName, 
                            cdmVersion = cdmVersion,
                            numThreads = numThreads,
                            sqlOnly = sqlOnly, 
                            sqlOnlyUnionCount = sqlOnlyUnionCount,
                            sqlOnlyIncrementalInsert = sqlOnlyIncrementalInsert,
                            outputFolder = outputFolder,
                            outputFile = outputFile,
                            verboseMode = verboseMode,
                            writeToTable = writeToTable,
                            writeToCsv = writeToCsv,
                            csvFile = csvFile,
                            checkLevels = checkLevels,
                checkSeverity = checkSeverity,
                            tablesToExclude = tablesToExclude,
                            checkNames = checkNames)

# inspect logs ----------------------------------------------------------------------------
ParallelLogger::launchLogViewer(logFileName = file.path(outputFolder, cdmSourceName, 
                                                      sprintf("log_DqDashboard_%s.txt", cdmSourceName)))

# (OPTIONAL) if you want to write the JSON file to the results table separately -----------------------------
jsonFilePath <- ""
DataQualityDashboard::writeJsonResultsToTable(connectionDetails = connectionDetails, 
                                            resultsDatabaseSchema = resultsDatabaseSchema, 
                                            jsonFilePath = jsonFilePath)
                                            